In [1]:
# Get the library files
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
dataset = pd.read_csv("DillibabuSarva_DefectDataset.csv")

In [3]:
# Display the dataset
dataset

,SHA,cbo,wmc,dit,rfc,lcom,totalMethods,totalFields,nosi,loc,...,tryCatchQty,parenthesizedExpsQty,stringLiteralsQty,numbersQty,assignmentsQty,mathOperationsQty,variablesQty,maxNestedBlocks,uniqueWordsQty,defect
0,7a955fd6c7de2bd912be544dcfe77f9173a7aa600,5,60,2,55,189,27,5,30,247,...,4,2,47,9,27,5,17,3,191,0
1,000f1ab4780fc9460975791c52597f7c04e15be70,3,10,1,1,9,7,4,1,38,...,0,0,0,22,4,0,4,2,69,0
2,000f1ab4780fc9460975791c52597f7c04e15be71,3,10,1,1,9,7,4,0,38,...,0,0,0,22,4,0,4,2,69,1
3,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c270,20,59,3,63,189,24,9,4,262,...,0,6,6,14,45,8,41,4,222,0
4,0024dbdd6ba3cc7797cc0b1ae537dcdc488c4c271,21,58,2,61,189,24,9,0,260,...,0,6,6,14,45,8,41,4,222,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6047,ffd1ed788cbf10bed00d49d79c7ee44250c36ac11,52,124,12,144,963,110,9,0,804,...,0,0,26,16,32,4,30,6,689,1
6048,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c0,24,27,2,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,0
6049,ffdf4a3fcccb7489548c6a2ff6af7cebc92a180c1,22,27,1,46,4,8,8,0,126,...,0,1,3,14,27,0,24,3,108,1
6050,ffe7c9989a4553d35fd1d5041d0cece0a673a0c80,3,12,2,12,28,8,0,1,67,...,2,0,0,2,10,0,8,2,36,0


In [4]:
# Display the column names
dataset.columns

Index(['SHA', 'cbo', 'wmc', 'dit', 'rfc', 'lcom', 'totalMethods',
       'totalFields', 'nosi', 'loc', 'returnQty', 'loopQty', 'comparisonsQty',
       'tryCatchQty', 'parenthesizedExpsQty', 'stringLiteralsQty',
       'numbersQty', 'assignmentsQty', 'mathOperationsQty', 'variablesQty',
       'maxNestedBlocks', 'uniqueWordsQty', 'defect'],
      dtype='object')

In [5]:
# Feature selection result
selected_features = ['nosi','dit','cbo','rfc','maxNestedBlocks',
                     'uniqueWordsQty','assignmentsQty','numbersQty',
                     'tryCatchQty','parenthesizedExpsQty']

X = dataset[selected_features]
y = dataset['defect']

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [7]:
from sklearn.svm import SVC
classifier = SVC(kernel = 'rbf', gamma = 'scale', C = 1.0, verbose = 3)
#fitting the model for grid search
classifier.fit(X_train, y_train)

[LibSVM]

SVC(verbose=3)

In [8]:
y_pred = classifier.predict(X_test)

In [9]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

In [10]:
print(cm)

[[246 662]
 [ 70 838]]


In [11]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred)

In [12]:
# SVClassification Report
print(clf_report)

              precision    recall  f1-score   support

           0       0.78      0.27      0.40       908
           1       0.56      0.92      0.70       908

    accuracy                           0.60      1816
   macro avg       0.67      0.60      0.55      1816
weighted avg       0.67      0.60      0.55      1816



In [13]:
# Finding the outliers for our dataset
Q1 = dataset[selected_features].quantile(0.25)
Q3 = dataset[selected_features].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
#Check
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 967
dit: Lesser = 0, Greater = 788
cbo: Lesser = 0, Greater = 403
rfc: Lesser = 0, Greater = 367
maxNestedBlocks: Lesser = 0, Greater = 411
uniqueWordsQty: Lesser = 0, Greater = 431
assignmentsQty: Lesser = 0, Greater = 490
numbersQty: Lesser = 0, Greater = 651
tryCatchQty: Lesser = 0, Greater = 651
parenthesizedExpsQty: Lesser = 0, Greater = 671


In [14]:
# Replacing the outliers with mean values
for col in selected_features:
    mean = dataset[col].mean()
    dataset[col] = np.where((dataset[col] > upper[col]) | (dataset[col] < lower[col]), mean, dataset[col])

In [15]:
# After replacing the outliers
for col in selected_features:
    less = (dataset[col] < lower[col]).sum()
    great = (dataset[col] > upper[col]).sum()
    print(f"{col}: Lesser = {less}, Greater = {great}")

nosi: Lesser = 0, Greater = 0
dit: Lesser = 0, Greater = 0
cbo: Lesser = 0, Greater = 0
rfc: Lesser = 0, Greater = 0
maxNestedBlocks: Lesser = 0, Greater = 0
uniqueWordsQty: Lesser = 0, Greater = 0
assignmentsQty: Lesser = 0, Greater = 0
numbersQty: Lesser = 0, Greater = 0
tryCatchQty: Lesser = 0, Greater = 0
parenthesizedExpsQty: Lesser = 0, Greater = 0


In [16]:
A = dataset[selected_features]
b = dataset['defect']
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size = 0.3, random_state = 42, stratify = b)

In [17]:
classifier_recheck = SVC(kernel = 'rbf', gamma = 'scale', C = 1.0, verbose = 3)
# fitting the model for grid search
classifier_recheck.fit(A_train, b_train)

[LibSVM]

SVC(verbose=3)

In [18]:
b_pred = classifier_recheck.predict(A_test)

In [19]:
cmodel = confusion_matrix(b_test,b_pred)
print(cmodel)

[[459 449]
 [196 712]]


In [20]:
# SVClassification Report after replacing outliers
clf_report_check = classification_report(b_test, b_pred)
print(clf_report_check)

              precision    recall  f1-score   support

           0       0.70      0.51      0.59       908
           1       0.61      0.78      0.69       908

    accuracy                           0.64      1816
   macro avg       0.66      0.64      0.64      1816
weighted avg       0.66      0.64      0.64      1816



In [21]:
# Here, we could see accuracy increased by 4%

In [24]:
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV
svc = SVC(random_state=42)
param_grid = {
    'kernel': ['rbf'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

In [25]:
grid_search_svc = GridSearchCV(
    estimator=svc, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)
grid_search_svc.fit(A_train, b_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100],
                         'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
                         'kernel': ['rbf']},
             scoring='accuracy')

In [26]:
Create a synthetic dataset
A, b = make_classification(n_samples=1000, n_features=20, random_state=42)
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size=0.2, random_state=42)

In [27]:
svc = SVC(random_state=42)

In [29]:
# Fit the grid search to the training data
grid_search_svc.fit(A_train, b_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100],
                         'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
                         'kernel': ['rbf']},
             scoring='accuracy')

In [30]:
# Convert the cv_results_ dictionary into a pandas DataFrame
results_df = pd.DataFrame(grid_search_svc.cv_results_)

In [32]:
# View the top 5 performing parameter combinations
display(results_df.sort_values(by='mean_test_score', ascending=False).head())

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
8,0.010090,0.001203,0.006547,0.000577,1.0,0.01,rbf,"{'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}",0.85000,0.89375,0.87500,0.86875,0.87500,0.87250,0.014031,1
17,0.012229,0.001204,0.005504,0.000651,100.0,0.001,rbf,"{'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}",0.85625,0.87500,0.86250,0.86250,0.89375,0.87000,0.013346,2
12,0.010111,0.001025,0.005804,0.000676,10.0,0.001,rbf,"{'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}",0.85625,0.88125,0.86875,0.87500,0.86875,0.87000,0.008292,2
0,0.020564,0.002128,0.084406,0.002945,0.1,scale,rbf,"{'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}",0.83750,0.85625,0.86875,0.88125,0.88750,0.86625,0.017941,4
3,0.017214,0.006408,0.011513,0.002533,0.1,0.01,rbf,"{'C': 0.1, 'gamma': 0.01, 'kernel': 'rbf'}",0.84375,0.85625,0.87500,0.87500,0.88125,0.86625,0.014031,4


In [33]:
# Extract and evaluate results
print(f"Best Hyperparameters: {grid_search_svc.best_params_}")
print(f"Best Cross-Validation Score: {grid_search_svc.best_score_:.4f}")

Best Hyperparameters: {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
Best Cross-Validation Score: 0.8725


In [34]:
# Evaluate performance on the untouched test set
best_model = grid_search_svc.best_estimator_
test_accuracy = best_model.score(A_test, b_test)
print(f"Test Set Accuracy: {test_accuracy:.4f}")

Test Set Accuracy: 0.8550


In [35]:
#importing pickle library for deployment phase
import pickle

In [36]:
#here we are assigning the saving model file name with extension to the filename variable
filename = "finalized-model_SV_Classification_Defect_Prediction.sav"
#using pickle dump method we are writing the file on disk
pickle.dump(classifier, open(filename, 'wb'))

In [37]:
import warnings
warnings.filterwarnings("ignore")
with open("finalized-model_SV_Classification_Defect_Prediction.sav", "rb") as f:
    model = pickle.load(f)
new_prediction = model.predict([[10, 5, 20, 70, 5, 200, 60, 30, 3, 10]])
print(f"Defect? {new_prediction[0]}")

Defect? 0


In [ ]:
# Here after removing outliers, we could see accuracy we are getting as 85.5%